In [1]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             precision_score, recall_score, f1_score, average_precision_score)

In [2]:
with open("rf_model.pkl", "rb") as f:
    bundle = pickle.load(f)

model = bundle["model"]
FEATURES = bundle["features"]
THRESHOLD = bundle["threshold"]

print("Threshold:", THRESHOLD)
print("Best settings:", bundle["best_params"])

Threshold: 0.6
Best settings: {'rf__n_estimators': 200, 'rf__min_samples_leaf': 10, 'rf__max_features': 0.5, 'rf__max_depth': 15, 'rf__class_weight': 'balanced'}


In [3]:
DATA_DIR = Path("../logistic_regression_modeling/final_data")

df = pd.read_csv(DATA_DIR / "cleaned_2025.csv")
df = df.iloc[:, :-4]
df = df[[i for i in df.columns if 'income' not in i]]
df = df[[i for i in df.columns if 'original' not in i]]
df = df[[i for i in df.columns if 'qa' not in i]]
df = df.drop(columns=['fd_label', 'label_status', 'label_reason'])

print(df.shape)
df.head()

(84119, 32)


,tract_id,State,County,meta_dataset_version,meta_population_census_year,meta_tract_vintage,clean__Urban,clean__population_total,clean__occupied_housing_units_total,clean__GroupQuartersFlag,...,clean__PovertyRate,clean__MedianFamilyIncome,clean__TractLOWI,clean__TractHUNV,clean__TractSNAP,clean__LATracts_half,clean__LATracts1,clean__LATractsVehicle_20,clean__low_access_population_1_20,FD_2
0,1001020100,Alabama,Autauga County,2025,2020,2020,1.0,1775.0,646.0,0.0,...,17.4,76979.0,1316.0,35.0,100.0,1.0,1.0,0.0,1080.0,1.0
1,1001020200,Alabama,Autauga County,2025,2020,2020,1.0,2055.0,723.0,0.0,...,9.5,62331.0,1148.0,28.0,143.0,1.0,0.0,0.0,167.0,0.0
2,1001020300,Alabama,Autauga County,2025,2020,2020,1.0,3216.0,1291.0,0.0,...,7.1,84191.0,2350.0,84.0,111.0,1.0,1.0,0.0,612.0,1.0
3,1001020400,Alabama,Autauga County,2025,2020,2020,1.0,4246.0,1707.0,0.0,...,15.3,81136.0,2580.0,114.0,78.0,1.0,0.0,0.0,14.0,0.0
4,1001020501,Alabama,Autauga County,2025,2020,2020,1.0,4322.0,1778.0,0.0,...,3.5,108233.0,4114.0,9.0,53.0,1.0,0.0,0.0,102.0,0.0


In [4]:
df.isna().sum()

tract_id                                  0
State                                     0
County                                    0
meta_dataset_version                      0
meta_population_census_year               0
meta_tract_vintage                        0
clean__Urban                              0
clean__population_total                   0
clean__occupied_housing_units_total       0
clean__GroupQuartersFlag                271
clean__NUMGQTRS                           0
clean__PCTGQTRS                         271
clean__TractKids                          0
clean__TractSeniors                       0
clean__TractWhite                         0
clean__TractBlack                         0
clean__TractAsian                         0
clean__TractNHOPI                         0
clean__TractAIAN                          0
clean__TractOMultir                       0
clean__TractHispanic                      0
clean__LowIncomeTracts                    0
clean__PovertyRate              

In [5]:
X_2025 = df[FEATURES]
probs = model.predict_proba(X_2025)[:, 1]

df["pred_prob"] = probs
df["pred_class"] = (probs >= THRESHOLD).astype(int)

df["tract_id"] = df["tract_id"].astype("int64").astype(str).str.zfill(11)

FIPS_TO_STATE = {
    "01": "AL", "02": "AK", "04": "AZ", "05": "AR", "06": "CA", "08": "CO",
    "09": "CT", "10": "DE", "11": "DC", "12": "FL", "13": "GA", "15": "HI",
    "16": "ID", "17": "IL", "18": "IN", "19": "IA", "20": "KS", "21": "KY",
    "22": "LA", "23": "ME", "24": "MD", "25": "MA", "26": "MI", "27": "MN",
    "28": "MS", "29": "MO", "30": "MT", "31": "NE", "32": "NV", "33": "NH",
    "34": "NJ", "35": "NM", "36": "NY", "37": "NC", "38": "ND", "39": "OH",
    "40": "OK", "41": "OR", "42": "PA", "44": "RI", "45": "SC", "46": "SD",
    "47": "TN", "48": "TX", "49": "UT", "50": "VT", "51": "VA", "53": "WA",
    "54": "WV", "55": "WI", "56": "WY", "72": "PR",
}
df = df.rename(columns={"State": "state_name"})
df["state"] = df["tract_id"].str[:2].map(FIPS_TO_STATE)

# risk: lowercase low / mid / high
df["risk"] = pd.cut(
    df["pred_prob"],
    bins=[0, 0.33, 0.67, 1],
    labels=["low", "mid", "high"],
    include_lowest=True,
).astype(str)

print(df["risk"].value_counts())
print("Unmapped states:", df["state"].isna().sum())

risk
low     47510
mid     23790
high    12819
Name: count, dtype: int64
Unmapped states: 0


In [6]:
labeled = df["FD_2"].notna()
y_true = df.loc[labeled, "FD_2"].astype(int)
y_pred = df.loc[labeled, "pred_class"]

print(f"Evaluating on {labeled.sum():,} labeled tracts ({(~labeled).sum()} unlabeled skipped)")
print("Accuracy :", accuracy_score(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))  # rows = actual, cols = predicted
print()
print(classification_report(y_true, y_pred, digits=3))

Evaluating on 83,848 labeled tracts (271 unlabeled skipped)
Accuracy : 0.8500977960118309
[[61893  8600]
 [ 3969  9386]]

              precision    recall  f1-score   support

           0      0.940     0.878     0.908     70493
           1      0.522     0.703     0.599     13355

    accuracy                          0.850     83848
   macro avg      0.731     0.790     0.753     83848
weighted avg      0.873     0.850     0.859     83848



In [7]:
lr = pd.read_csv("../logistic_regression_modeling/cleaned_2025_w_predictions.csv",
                 usecols=["tract_id", "pred_prob", "pred_class"])
lr["tract_id"] = lr["tract_id"].astype("int64").astype(str).str.zfill(11)
lr = lr.rename(columns={"pred_prob": "lr_prob", "pred_class": "lr_class"})

compare = (df.loc[labeled, ["tract_id", "FD_2", "pred_prob", "pred_class"]]
             .merge(lr, on="tract_id", how="inner"))
y = compare["FD_2"].astype(int)

def scores(y_true, y_class, y_prob):
    return {
        "accuracy":  accuracy_score(y_true, y_class),
        "precision": precision_score(y_true, y_class, zero_division=0),
        "recall":    recall_score(y_true, y_class),
        "f1":        f1_score(y_true, y_class),
        "avg_precision": average_precision_score(y_true, y_prob),
    }

comparison = pd.DataFrame({
    "Logistic regression": scores(y, compare["lr_class"], compare["lr_prob"]),
    "Random forest":       scores(y, compare["pred_class"], compare["pred_prob"]),
}).round(3)

print(f"Compared on {len(compare):,} tracts")
comparison

Compared on 83,848 tracts


,Logistic regression,Random forest
accuracy,0.832,0.850
precision,0.393,0.522
recall,0.106,0.703
f1,0.166,0.599
avg_precision,0.323,0.624


In [8]:
df.to_csv("cleaned_2025_w_rf_predictions.csv", index=False)
print("Saved cleaned_2025_w_rf_predictions.csv")

Saved cleaned_2025_w_rf_predictions.csv
